In [61]:
import pandas as pd
import numpy as np
import sys
import warnings
warnings.filterwarnings('ignore') # Tắt các cảnh báo

sys.path.append('..')  # Thêm thư mục cha để import get_data_new
from get_data_new import get_client, get_datbase, detail_info, fetch_qmdata
import plotly.express as px
import pymongo
from get_data_new import URI_MONGODB, detail_info
from statsmodels.tsa.stattools import grangercausalitytests, adfuller

# Granger Test

In [62]:
def calc_market_breadth(df: pd.DataFrame, threshold = 0.005) -> pd.DataFrame:
    """
    Tính Market Breadth (Advances, Declines, Unchanged) sử dụng vectorized operation.
    
    Args:
        df (pd.DataFrame): DataFrame chứa tỷ suất sinh lợi (returns) của các cổ phiếu. 
                          Index là ngày, Columns là mã cổ phiếu.
        threshold (float): Ngưỡng để xác định Unchanged. Mặc định là 0.0.
                          Nếu |return| <= threshold thì coi là Unchanged.
    
    Returns:
        pd.DataFrame: DataFrame chứa 3 cột: Advances, Declines, Unchanged theo từng ngày.
    """

    advances = (df > threshold).sum(axis=1)
    declines = (df < -threshold).sum(axis=1)
    unchanged = ((df >= -threshold) & (df <= threshold)).sum(axis=1)
    
    breadth_df = pd.DataFrame({
        'Advances': advances,
        'Declines': declines,
        'Unchanged': unchanged
    }, index=df.index)
    
    return breadth_df

In [63]:
market_breadth_6y = pd.read_csv('market_breadth_6y.csv', index_col=0, parse_dates=True)
market_breadth_6y

,Advance,Decline,Unchanged
datetime,,,
2020-01-02,178,144,199
2020-01-03,133,165,223
2020-01-06,92,227,202
2020-01-07,142,132,247
2020-01-08,76,246,199
...,...,...,...
2026-03-27,407,149,243
2026-03-30,223,325,272
2026-03-31,277,235,303


In [64]:
# chuyển mỗi cột thành dạng percentage của tổng các cột tại từng row
market_breadth_pct_6y = market_breadth_6y.div(market_breadth_6y.sum(axis=1), axis=0) * 100
market_breadth_pct_6y

,Advance,Decline,Unchanged
datetime,,,
2020-01-02,34.165067,27.639155,38.195777
2020-01-03,25.527831,31.669866,42.802303
2020-01-06,17.658349,43.570058,38.771593
2020-01-07,27.255278,25.335893,47.408829
2020-01-08,14.587332,47.216891,38.195777
...,...,...,...
2026-03-27,50.938673,18.648310,30.413016
2026-03-30,27.195122,39.634146,33.170732
2026-03-31,33.987730,28.834356,37.177914


In [65]:
db = pymongo.MongoClient(URI_MONGODB)
mongo_data = db['data']['qmdata']
hose_stock_symbols = detail_info(mongo_data, info_need='symbol', exchange='HOSE', kind='stock')
vnindex_6y = fetch_qmdata(symbol='VNINDEX', interval='1D', exchange = 'HOSE', start = '2020/01/02', stop='now',kind='index', division='')

In [67]:
vnindex_6y_returns = vnindex_6y['close'].pct_change().dropna()

In [68]:
def test_adf_stationarity(feature: pd.Series, max_diff=5, max_lag=10):
    """
    Kiểm định ADF để tìm bậc sai phân (integration order).
    
    - Thử từ level (d=0) đến sai phân bậc max_diff
    - Nếu tại bậc nào p-value < 0.05 → chuỗi dừng tại đó
    - Nếu đến max_diff vẫn không dừng → trả về None
    """
    
    feature = feature.dropna()
    
    for d in range(0, max_diff + 1):
        series = feature.copy()
        
        # Áp dụng sai phân d lần
        for _ in range(d):
            series = series.diff().dropna()
        
        best_p = 1.0
        
        # Test với nhiều lag để ổn định kết quả
        for lag in range(0, max_lag + 1):
            try:
                result = adfuller(series, maxlag=max_lag, autolag='AIC')
                p_value = result[1]
                
                if p_value < 0.05:
                    return {
                        "integration_order": d,
                        "ADF p-value": p_value,
                        "lag_used": result[2]
                    }
                
                best_p = min(best_p, p_value)
            
            except Exception:
                continue
    
    # Nếu không dừng đến bậc max_diff
    return {
        "integration_order": None,
        "ADF p-value": None,
        "lag_used": None
    }

In [69]:
for col in market_breadth_6y.columns:
    result = test_adf_stationarity(market_breadth_6y[col])
    print(f"ADF Stationarity Test for {col}: {result}")

ADF Stationarity Test for Advance: {'integration_order': 0, 'ADF p-value': np.float64(2.9842287020869137e-07), 'lag_used': 10}
ADF Stationarity Test for Decline: {'integration_order': 0, 'ADF p-value': np.float64(1.1236719497941266e-09), 'lag_used': 9}
ADF Stationarity Test for Unchanged: {'integration_order': 0, 'ADF p-value': np.float64(0.0480144753797224), 'lag_used': 9}


In [70]:
for col in market_breadth_pct_6y.columns:
    result = test_adf_stationarity(market_breadth_pct_6y[col])
    print(f"ADF Stationary test for Percentage {col}: {result}")


ADF Stationary test for Percentage Advance: {'integration_order': 0, 'ADF p-value': np.float64(6.041059359719892e-15), 'lag_used': 9}
ADF Stationary test for Percentage Decline: {'integration_order': 0, 'ADF p-value': np.float64(8.044495216018645e-16), 'lag_used': 9}
ADF Stationary test for Percentage Unchanged: {'integration_order': 0, 'ADF p-value': np.float64(0.0005535306049320055), 'lag_used': 9}


In [71]:
result = test_adf_stationarity(vnindex_6y['close'])
print(f"ADF Stationarity Test for VNIndex: {result}")
result = test_adf_stationarity(vnindex_6y_returns)
print(f"ADF Stationarity Test for VNIndex Returns: {result}")

ADF Stationarity Test for VNIndex: {'integration_order': 1, 'ADF p-value': np.float64(6.09487508289223e-29), 'lag_used': 5}


In [95]:
def test_granger_causality(x: pd.Series, y: pd.Series, max_lag: int, alpha: float = 0.05):
    df = pd.DataFrame({'target': y, 'feature': x}).dropna()
    
    results = grangercausalitytests(df, maxlag=max_lag, verbose=False)
    
    significant_lags = []
    
    for lag in range(1, max_lag + 1):
        try:
            p_value = results[lag][0]['ssr_ftest'][1]
            
            if p_value < alpha:
                significant_lags.append({
                    "lag": lag,
                    "p_value": p_value
                })
                
        except Exception:
            continue
    
    # sort theo p-value tăng dần (optional nhưng nên có)
    significant_lags = sorted(significant_lags, key=lambda x: x["p_value"])
    
    return {
        "causal": len(significant_lags) > 0,
        "significant_lags": significant_lags
    }

test_result1 = test_granger_causality(market_breadth_6y['Advance'],vnindex_6y['close'], max_lag=20)
test_result2 = test_granger_causality(market_breadth_6y['Decline'],vnindex_6y['close'], max_lag=20)

print("Granger Causality Test with Significant Lags - Advance -> VNIndex:", test_result1)
print("Granger Causality Test with Significant Lags - Decline -> VNIndex:", test_result2)

Granger Causality Test with Significant Lags - Advance -> VNIndex: {'causal': True, 'significant_lags': [{'lag': 5, 'p_value': np.float64(0.0034561582067882987)}, {'lag': 15, 'p_value': np.float64(0.0113674947609808)}, {'lag': 14, 'p_value': np.float64(0.015473179691569491)}, {'lag': 1, 'p_value': np.float64(0.018313476312549138)}, {'lag': 16, 'p_value': np.float64(0.027481781309702715)}, {'lag': 19, 'p_value': np.float64(0.030328403137140792)}, {'lag': 17, 'p_value': np.float64(0.03577546595682673)}, {'lag': 13, 'p_value': np.float64(0.03765672183292581)}, {'lag': 18, 'p_value': np.float64(0.04202877292398618)}, {'lag': 12, 'p_value': np.float64(0.042254178206245094)}]}
Granger Causality Test with Significant Lags - Decline -> VNIndex: {'causal': True, 'significant_lags': [{'lag': 1, 'p_value': np.float64(0.012290452128435218)}, {'lag': 5, 'p_value': np.float64(0.024383537280142288)}]}


In [96]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

def granger_single_lag(feature: pd.Series, target: pd.Series, max_lag=20, target_lag=5):
    """
    Test Granger causality từng lag riêng lẻ.
    
    feature: biến giải thích (X)
    target: biến mục tiêu (Y)
    max_lag: số lag tối đa của feature
    target_lag: số lag của target để control
    """
    
    df = pd.concat([target, feature], axis=1)
    df.columns = ['target', 'feature']
    df = df.replace([np.inf, -np.inf], np.nan).dropna()
    
    results = {}
    
    for lag in range(1, max_lag + 1):
        temp = df.copy()
        
        # tạo lag cho target (control)
        for l in range(1, target_lag + 1):
            temp[f"target_lag_{l}"] = temp['target'].shift(l)
        
        # tạo lag riêng cho feature
        temp[f"feature_lag_{lag}"] = temp['feature'].shift(lag)
        
        temp = temp.dropna()
        
        # check dữ liệu đủ
        if len(temp) < 30:
            results[f"lag_{lag}"] = 1.0
            continue
        
        # define X, y
        X_cols = [f"target_lag_{l}" for l in range(1, target_lag + 1)]
        X_cols.append(f"feature_lag_{lag}")
        
        X = temp[X_cols]
        y = temp['target']
        
        X = sm.add_constant(X)
        
        try:
            model = sm.OLS(y, X).fit()
            p_value = model.pvalues[f"feature_lag_{lag}"]
        except Exception:
            p_value = 1.0
        
        results[f"lag_{lag}"] = p_value
    
    # tìm lag tốt nhất
    best_lag = min(results, key=results.get)
    best_p = results[best_lag]
    
    return {
        "lag_p_values": results,
        "best_lag": best_lag,
        "best_p_value": best_p
    }

result = granger_single_lag(market_breadth_pct_6y['Advance'], vnindex_6y_returns, max_lag=15, target_lag=15)
print("Granger Single Lag Test - Advance% -> VNIndex:", result['best_lag'], "with p-value:", result['best_p_value'])
result2 = granger_single_lag(market_breadth_pct_6y['Decline'], vnindex_6y_returns, max_lag=5, target_lag=5)
print("Granger Single Lag Test - Decline% -> VNIndex:", result2['best_lag'], "with p-value:", result2['best_p_value'])

Granger Single Lag Test - Advance% -> VNIndex: lag_11 with p-value: 0.18777518771020713
Granger Single Lag Test - Decline% -> VNIndex: lag_3 with p-value: 0.3762297809306069


In [99]:
ma_windows = [5, 15]

advance_ma = pd.DataFrame({
    f"Advance_MA_{w}": market_breadth_pct_6y["Advance"].rolling(w).mean()
    for w in ma_windows
}, index=market_breadth_6y.index)

advance_ma = advance_ma.dropna()
advance_ma

,Advance_MA_5,Advance_MA_15
datetime,,
2020-01-22,32.413793,27.708405
2020-01-30,30.727969,26.605702
2020-01-31,28.390805,25.810616
2020-02-03,25.440613,25.463533
2020-02-04,23.678161,25.651623
...,...,...
2026-03-27,41.114959,36.321722
2026-03-30,44.490289,37.828402
2026-03-31,39.192477,35.775444


In [101]:
for i in range(len(ma_windows)):
    test_result = test_granger_causality(advance_ma[f"Advance_MA_{ma_windows[i]}"], vnindex_6y['close'], max_lag=20)
    print(f"Granger Causality Test - Advance_MA_{ma_windows[i]} -> VNIndex:", test_result)

Granger Causality Test - Advance_MA_5 -> VNIndex: {'causal': np.False_, 'lag': 11, 'p_value': np.float64(0.13839227067835586)}
Granger Causality Test - Advance_MA_15 -> VNIndex: {'causal': np.True_, 'lag': 20, 'p_value': np.float64(0.032977727723080996)}


# Plot figure

In [102]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
vnindex_6y_ohlcv = vnindex_6y.loc['2020-01-01':, ['open', 'high', 'low', 'close', 'volume']]
vnindex_6y_ohlc = vnindex_6y_ohlcv.drop(columns=['volume'])

## Advance and Advance SMA(5,15) vs VNINDEX

In [113]:
import plotly.graph_objects as go

fig = go.Figure()

# Candlestick (y-axis chính)
fig.add_trace(go.Candlestick( x=vnindex_6y_ohlc.index,open=vnindex_6y_ohlc['open'],high=vnindex_6y_ohlc['high'],low=vnindex_6y_ohlc['low'], close=vnindex_6y_ohlc['close'],name='VNIndex'))

# Advance% (y-axis phụ)
# fig.add_trace(go.Scatter(
#     x=vnindex_6y_ohlc.index,
#     y=market_breadth_pct_6y['Advance'],
#     mode='lines',
#     name='Advance%',
#     yaxis='y2'
# ))

# Moving Average (cũng y-axis phụ)
fig.add_trace(go.Scatter(
    x=advance_ma.index,
    y=advance_ma[f"Advance_MA_{ma_windows[0]}"],
    mode='lines',
    name=f"Advance_MA_{ma_windows[0]}",
    yaxis='y2',
    line=dict(color='blue', width=2),
    opacity=0.8
))

fig.add_trace(go.Scatter(
    x=advance_ma.index,
    y=advance_ma[f"Advance_MA_{ma_windows[1]}"],
    mode='lines',
    name=f"Advance_MA_{ma_windows[1]}",
    yaxis='y2',
    line=dict(color='purple', width=2),
    opacity=0.8
))

# Layout với 2 trục y
fig.update_layout(
    title='VNIndex with Advance% Moving Averages',
    xaxis=dict(title='Date'),
    yaxis=dict( title='VNIndex'),
    yaxis2=dict(  # trục phải
        title='Advance%',
        overlaying='y',   # chồng lên cùng plot
        side='right'
    )
)

fig.update(layout_xaxis_rangeslider_visible=False)

fig.show()

In [ ]:

# Tạo 2 subplot (2 hàng, 1 cột)
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,   # sync zoom trục x
    vertical_spacing=0.05,
    row_heights=[0.7, 0.3]  # VNIndex to hơn
)

# --- Plot 1: VNIndex (OHLC) ---
fig.add_trace(go.Candlestick(
    x=vnindex_6y_ohlc.index,
    open=vnindex_6y_ohlc['open'],
    high=vnindex_6y_ohlc['high'],
    low=vnindex_6y_ohlc['low'],
    close=vnindex_6y_ohlc['close'],
    name='VNIndex'
), row=1, col=1)

# --- Plot 2: Advance% ---
# fig.add_trace(go.Scatter(
#     x=vnindex_6y_ohlc.index,
#     y=market_breadth_pct_6y['Advance'],
#     mode='lines',
#     name='Advance%',
#     opacity=0.3,
# ), row=2, col=1)

# MA
fig.add_trace(go.Scatter(
    x=advance_ma.index,
    y=advance_ma[f"Advance_MA_{ma_windows[0]}"],
    mode='lines',
    name=f"MA {ma_windows[0]}"
), row=2, col=1)

fig.add_trace(go.Scatter(
    x=advance_ma.index,
    y=advance_ma[f"Advance_MA_{ma_windows[1]}"],
    mode='lines',
    name=f"MA {ma_windows[1]}"
), row=2, col=1)

# Layout
fig.update_layout(
    title='VNIndex & Market Breadth (Advance%)',
    width=1400,
    height=800,
    xaxis= dict( range = [vnindex_6y_ohlc.index[-252], vnindex_6y_ohlc.index[-1]]),
    yaxis_title='VNIndex',
    yaxis2_title='Advance%'
)
fig.update_layout(
    hovermode='x unified'
)

fig.update(layout_xaxis_rangeslider_visible=False)

fig.show()

In [ ]:


fig = make_subplots(
    rows=4, cols=1,
    shared_xaxes=True,
    subplot_titles=("VNIndex OHLC", "Advance", "Decline", "Unchanged"),
    vertical_spacing=0.06,
    row_heights=[0.4, 0.2, 0.2, 0.2]  # OHLC to hơn 3 subplot còn lại
)

# --- Subplot 1: OHLC ---
fig.add_trace(
    go.Ohlc(
        x=vnindex_6y_ohlc.index,
        open=vnindex_6y_ohlc['open'],
        high=vnindex_6y_ohlc['high'],
        low=vnindex_6y_ohlc['low'],
        close=vnindex_6y_ohlc['close'],
        name='VNIndex'
    ),
    row=1, col=1
)

# --- Subplot 2, 3, 4: Advance / Decline / Unchanged ---
breadth_metrics = ['Advance', 'Decline', 'Unchanged']
colors = ['green', 'red', 'gray']

for i, (metric, color) in enumerate(zip(breadth_metrics, colors)):
    fig.add_trace(
        go.Bar(
            x=market_breadth_pct_6y.index,
            y = market_breadth_pct_6y[metric],
            name=metric,
            marker_color=color,
            opacity=1
        ),
        row=i+2, col=1
    )
#Add advance MA(5) lines vào subplot số 2
fig.add_trace(
        go.Scatter(
            x=advance_ma.index,
            y=advance_ma['Advance_MA_10'],
            mode='lines',
            name='Advance MA(10)',
            line=dict(color='blue', width=2)
         ),
         row=2, col=1
     )
# Tắt rangeslider
fig.update_xaxes(rangeslider_visible=False)
for row in [2, 3, 4]:
    fig.update_yaxes(range=[0, 100], row=row, col=1)
fig.update_layout(
    height=900,
    title_text="VNIndex & Market Breadth Analysis",
    showlegend=True,
    hovermode='x unified'  # Hover đồng bộ trên tất cả subplot
)

fig.show()

In [ ]:
import plotly.graph_objects as go

fig = go.Figure(data= go.Ohlc(x=vnindex_6y_ohlcv.index,
                              open=vnindex_6y_ohlcv['open'], 
                             high=vnindex_6y_ohlcv['high'], 
                             low=vnindex_6y_ohlcv['low'], 
                             close=vnindex_6y_ohlcv['close'],
                             name='VNIndex OHLC',
                             hovertext=vnindex_6y_ohlcv.index.strftime('%Y-%m-%d')))
fig.update(layout_xaxis_rangeslider_visible=False)
fig.update_layout(title='VNIndex OHLC Chart (2020-2026)', yaxis_title='Price')
fig.add_trace(go.Scatter(x=market_breadth_6y.index, y=market_breadth_6y['Advance'], mode='lines', name='Advance', yaxis='y2'))
fig.update_layout(yaxis2=dict(title='Advance Count', overlaying='y', side='right'))

fig.show()

In [ ]:
vnindex_6y_ohlcv.columns = ["Open", "High", "Low", "Close", "Volume"]


In [ ]:
import vectorbt as vbt
vnindex_6y_ohlcv.vbt.ohlcv.plot().show()